In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# document = PyPDFLoader('ml_paper1.pdf')
document = DirectoryLoader(
    path='docs',
    glob='*.pdf',
    loader_cls=PyPDFLoader
)
text_content = document.load()
# text_content = document.lazy_load()                               #for loading docs one-by-one from a directory..better for streaming
# for doc in text_content:
#     print(doc)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
content = text_splitter.split_documents(text_content)
content[1].page_content

'employed for numerous NLP tasks and provide a walk-through of their evolution. We also summarize, compare and contrast the\nvarious models and put forward a detailed understanding of the past, present and future of deep learning in NLP.\nIndex Terms\nNatural Language Processing, Deep Learning, Word2Vec, Attention, Recurrent Neural Networks, Convolutional Neural Net-\nworks, LSTM, Sentiment Analysis, Question Answering, Dialogue Systems, Parsing, Named-Entity Recognition, POS Tagging,\nSemantic Role Labeling\nI. I NTRODUCTION\nNatural language processing (NLP) is a theory-motivated range of computational techniques for the automatic analysis and\nrepresentation of human language. NLP research has evolved from the era of punch cards and batch processing, in which the\nanalysis of a sentence could take up to 7 minutes, to the era of Google and the likes of it, in which millions of webpages can'

In [21]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
import os
load_dotenv()
api_key=os.getenv('GOOGLE_API_KEY')
embeddings = GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-001',
    google_api_key=api_key
)
embeddings1 = OllamaEmbeddings(model='embeddinggemma')                  #open-source embedding model could be used too
db = FAISS.from_documents(content[:5], embeddings)

In [22]:
query = 'What is natural language processing?'
result = db.similarity_search(query)
result[1].page_content

'learning related models and methods applied to natural language tasks such as convolutional neural networks (CNNs), recurrent\nneural networks (RNNs), and recursive neural networks. We also discuss memory-augmenting strategies, attention mechanisms\nand how unsupervised models, reinforcement learning methods and recently, deep generative models have been employed for\nlanguage-related tasks.\nTo the best of our knowledge, this work is the ﬁrst of its type to comprehensively cover the most popular deep learning\nmethods in NLP research today 1. The work by Goldberg [6] only presented the basic principles for applying neural networks\nto NLP in a tutorial manner. We believe this paper will give readers a more comprehensive idea of current practices in this\ndomain.\nThe structure of the paper is as follows: Section II introduces the concept of distributed representation, the basis of'

In [ ]:
from langchain_ollama import OllamaLLM
# llm = OllamaLLM(model='gemma3:1b')
llm = OllamaLLM(model='qwen2.5:3b')                 #better chat models provide better answers from the context

In [30]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
Answer the following questions based on the provided context only. Think step-by-step before providing any
                                          answer. 1000 reward points will be awarded to you for every 
                                          correct answer.
                                          <context>
                                          {context}
                                          </context>
                                          Question:{input}
""")

In [31]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(llm, prompt)

In [32]:
retriever = db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002007C15DAF0>, search_kwargs={})

In [33]:
from langchain_classic.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever, document_chain)
response=retrieval_chain.invoke({"input": "Natural language processing (NLP) is a theory-motivated range of computational technique"})

In [34]:
response['answer']

'Natural Language Processing (NLP) is a theoretical-driven set of computational techniques aimed at automating the analysis and representation of human language. The evolution of NLP has been significant, moving from early stages characterized by tasks taking several minutes to analyze sentences (up to 7 minutes), to modern-day systems where millions of webpages can be processed in less than a second. This transformation is attributed largely to advancements in techniques such as Deep Learning, which have enabled the computer to perform various natural language-related tasks efficiently.\n\nThe context provided highlights that NLP has evolved significantly with deep learning methods, leveraging multiple layers for hierarchical data representation and achieving state-of-the-art results across various domains. Recent studies focus on incorporating new, dense vector-based models from deep learning into NLP tasks, improving performance in areas like machine translation, sentiment analysis,